# Préparation des données Divvy

Ce notebook présente le pipeline de préparation des données Divvy, de l'ingestion brute jusqu'à la production des tables analytiques et du graphe sémantique.

Les traitements sont organisés en quatre couches :
- **Bronze** : conservation des fichiers sources au format Delta.
- **Silver** : typage, déduplication et traitement des durées invalides.
- **Gold** : agrégations destinées à l'analyse des stations et des périodes.
- **Sémantique** : représentation des résultats dans un graphe RDF interrogeable en SPARQL.

La problématique étudiée est : *Dans quelles stations et à quelles périodes concentrer les campagnes de conversion des utilisateurs occasionnels ?*


In [6]:
import sys
sys.path.append("/home/jovyan/work")

from commun.spark_utils import (
    creer_session,
    creer_bucket,
    format_tables,
    RAW,
    LAKE
)

# Initialisation de Spark et préparation du stockage objet.
spark = creer_session("Divvy-Preparation")
format_tables(spark)
creer_bucket()

# Les dates sont interprétées dans le fuseau horaire local de Chicago.
spark.conf.set("spark.sql.session.timeZone", "America/Chicago")


Préparation de Spark 4.2.0 (Hadoop 3.5.0) + Delta Lake + data lake S3 … le premier démarrage peut prendre quelques minutes.
Session prête. Spark UI : http://localhost:4040
Bucket « datalake » déjà présent.


### 1.1. Lecture des fichiers sources

Les fichiers CSV des 24 mois étudiés (2024–2025) sont lus récursivement depuis le répertoire `RAW`. L'inférence automatique des types est désactivée : les valeurs sont d'abord conservées sous forme de texte dans la couche Bronze.


In [5]:
# Lecture de l'ensemble des fichiers CSV sans transformation métier.
divvy_raw = (
    spark.read
    .option("header", "true")
    .option("inferSchema", "false")
    .option("recursiveFileLookup", "true")
    .csv(RAW)
)

print(f"Nombre de lignes sources : {divvy_raw.count():,}")


Nombre de lignes sources : 11,413,562


### 1.2. Enregistrement dans le Data Lake

Les données sources sont enregistrées au format Delta dans RustFS, sous le chemin `bronze/divvy_trips`. Cette couche conserve les données brutes avant nettoyage.

Le mode `overwrite` remplace la version Bronze existante lors d'une nouvelle exécution complète.


In [7]:
# Écriture des données brutes dans la couche Bronze.
(
    divvy_raw.write
    .format("delta")
    .mode("overwrite")
    .save(f"{LAKE}/bronze/divvy_trips")
)


### 1.3. Rechargement de la couche Bronze

On recharge la table Delta qui servira de source au traitement Silver.


In [8]:
bronze = (
    spark.read
    .format("delta")
    .load(f"{LAKE}/bronze/divvy_trips")
)

print(f"Volume de la couche Bronze : {bronze.count():,} lignes")


Volume de la couche Bronze : 11,413,562 lignes


# 2. Couche Silver — Nettoyage et normalisation

## Objectif

La couche Silver prépare les données pour les analyses :
- conversion des dates et coordonnées vers des types adaptés ;
- calcul de la durée de chaque trajet ;
- suppression des doublons sur `ride_id` ;
- exclusion des durées nulles ou négatives, en tenant compte des changements d'heure à Chicago.

Les stations manquantes sont conservées : elles ne permettent pas une analyse par station, mais les trajets restent utiles pour les analyses temporelles et les répartitions globales.


### 2.1. Conversion des types et calcul de la durée

Les dates sont converties en timestamps et les coordonnées en nombres décimaux. La durée est calculée en secondes à partir des dates de début et de fin.


In [13]:
from pyspark.sql import functions as F

# Dates des changements d'heure d'automne à Chicago sur la période étudiée.
dates_changement_heure = ["2024-11-03", "2025-11-02"]

silver = (
    bronze
    .withColumn("started_at", F.to_timestamp("started_at"))
    .withColumn("ended_at", F.to_timestamp("ended_at"))
    .withColumn("start_lat", F.col("start_lat").cast("double"))
    .withColumn("start_lng", F.col("start_lng").cast("double"))
    .withColumn("end_lat", F.col("end_lat").cast("double"))
    .withColumn("end_lng", F.col("end_lng").cast("double"))
    .withColumn(
        "ride_duration_seconds",
        F.col("ended_at").cast("long") - F.col("started_at").cast("long")
    )
)


### 2.2. Déduplication et traitement des durées invalides

Les doublons sont supprimés en conservant une seule ligne par identifiant de trajet.

Les durées strictement positives sont conservées. Les durées négatives observées pendant les deux changements d'heure sont conservées, mais leur durée est remplacée par `NULL`, car l'ambiguïté liée au changement d'heure ne permet pas de calcul fiable. Les autres durées nulles ou négatives sont exclues.


In [14]:
# Une ligne unique est conservée pour chaque identifiant de trajet.
silver = silver.dropDuplicates(["ride_id"])

# Les durées négatives des changements d'heure sont conservées avec une durée inconnue.
# Les autres durées nulles ou négatives sont écartées.
silver = silver.filter(
    (F.col("ride_duration_seconds") > 0) |
    (
        F.to_date("started_at").isin(dates_changement_heure) &
        (F.col("ride_duration_seconds") < 0)
    )
)

silver = silver.withColumn(
    "ride_duration_seconds",
    F.when(F.col("ride_duration_seconds") < 0, None)
     .otherwise(F.col("ride_duration_seconds"))
)


### 2.3. Enregistrement de la couche Silver

Les données nettoyées sont enregistrées au format Delta dans RustFS. Cette table constitue la source des agrégations Gold.


In [15]:
(
    silver.write
    .format("delta")
    .mode("overwrite")
    .save(f"{LAKE}/silver/divvy_trips")
)


In [16]:
silver = (
    spark.read
    .format("delta")
    .load(f"{LAKE}/silver/divvy_trips")
)

print(f"Volume de la couche Silver : {silver.count():,} lignes")

Volume de la couche Silver : 11,412,671 lignes


La couche Silver contient **11 412 671 trajets**, contre **11 413 562** dans Bronze. Les 891 lignes écartées correspondent aux doublons et aux durées invalides hors changements d'heure. Les 72 durées négatives liées aux changements d'heure sont conservées avec une durée `NULL`.

Les informations de station manquantes sont volontairement conservées dans Silver ; elles seront exclues uniquement des agrégations qui nécessitent une station de départ identifiée.


# 3. Couche Gold — Agrégation et préparation décisionnelle

## Objectif

La couche Gold transforme les données Silver en tables synthétiques qui répondent à la problématique : *Dans quelles stations et à quelles périodes concentrer les campagnes de conversion des utilisateurs occasionnels ?*

Trois tables sont produites :
- `gold_station_period` : volume des trajets par station, année, mois, jour de semaine, heure et type de véhicule ;
- `gold_monthly_overview` : évolution mensuelle des trajets par catégorie d'utilisateur et type de véhicule ;
- `gold_station_ranking` : classement des stations selon le volume de trajets occasionnels et leur part dans le trafic.

Les analyses portent sur 2024 et 2025. Les agrégations par station utilisent uniquement les trajets dont la station de départ est renseignée.


### 3.1. Préparation des données pour les analyses par station

On limite la période à 2024–2025 et on extrait les dimensions temporelles nécessaires aux analyses : année, mois, jour de semaine et heure.


In [17]:
# Rechargement de la couche Silver comme source des agrégations.
silver = (
    spark.read
    .format("delta")
    .load(f"{LAKE}/silver/divvy_trips")
)

trajets_stations = (
    silver
    .filter(F.year("started_at").between(2024, 2025))
    .filter(
        F.col("start_station_id").isNotNull() &
        (F.trim(F.col("start_station_id")) != "")
    )
    .withColumn("year", F.year("started_at"))
    .withColumn("month", F.month("started_at"))
    .withColumn("day_of_week", F.dayofweek("started_at"))
    .withColumn("hour", F.hour("started_at"))
)


### 3.2. Agrégation par station et période

Pour chaque combinaison station/période/type de véhicule, on calcule le nombre total de trajets, le nombre de trajets occasionnels et abonnés, ainsi que la part des trajets occasionnels.


In [18]:
gold_station_period = (
    trajets_stations
    .groupBy(
        "start_station_id",
        "start_station_name",
        "year",
        "month",
        "day_of_week",
        "hour",
        "rideable_type"
    )
    .agg(
        F.count("*").alias("total_trips"),
        F.sum(F.when(F.col("member_casual") == "casual", 1).otherwise(0))
         .alias("casual_trips"),
        F.sum(F.when(F.col("member_casual") == "member", 1).otherwise(0))
         .alias("member_trips")
    )
    .withColumn(
        "casual_share_pct",
        F.round(F.col("casual_trips") / F.col("total_trips") * 100, 2)
    )
)

# Stockage de l'agrégation dans le Data Lake.
(
    gold_station_period.write
    .format("delta")
    .mode("overwrite")
    .save(f"{LAKE}/gold/station_period")
)


### 3.3. Évolution mensuelle des trajets

Cette table permet de comparer les volumes mensuels des utilisateurs occasionnels et abonnés, selon le type de véhicule. La durée moyenne est également calculée en secondes et en minutes.


In [19]:
trajets_mensuels = (
    silver
    .filter(F.year("started_at").between(2024, 2025))
    .withColumn("year", F.year("started_at"))
    .withColumn("month", F.month("started_at"))
)

gold_monthly_overview = (
    trajets_mensuels
    .groupBy("year", "month", "member_casual", "rideable_type")
    .agg(
        F.count("*").alias("total_trips"),
        F.avg("ride_duration_seconds").alias("avg_duration_seconds")
    )
    .withColumn(
        "avg_duration_minutes",
        F.round(F.col("avg_duration_seconds") / 60, 2)
    )
)

(
    gold_monthly_overview.write
    .format("delta")
    .mode("overwrite")
    .save(f"{LAKE}/gold/monthly_overview")
)


### 3.4. Classement des stations

Le classement est calculé à partir des trajets dont la station de départ est connue. Il fournit le volume total, le volume occasionnel, le volume abonné et la part occasionnelle par station.

Un seuil minimal de **500 trajets** sur la période est appliqué afin de limiter l'influence des stations très peu fréquentées.


In [20]:
gold_station_ranking = (
    trajets_stations
    .groupBy("start_station_id", "start_station_name")
    .agg(
        F.count("*").alias("total_trips"),
        F.sum(F.when(F.col("member_casual") == "casual", 1).otherwise(0))
         .alias("casual_trips"),
        F.sum(F.when(F.col("member_casual") == "member", 1).otherwise(0))
         .alias("member_trips")
    )
    .withColumn(
        "casual_share_pct",
        F.round(F.col("casual_trips") / F.col("total_trips") * 100, 2)
    )
    .filter(F.col("total_trips") >= 500)
)

(
    gold_station_ranking.write
    .format("delta")
    .mode("overwrite")
    .save(f"{LAKE}/gold/station_ranking")
)


In [21]:
# Vérification du nombre de lignes des trois tables Gold.

print(f"gold_station_period : {gold_station_period.count():,} lignes")
print(f"gold_monthly_overview : {gold_monthly_overview.count():,} lignes")
print(f"gold_station_ranking : {gold_station_ranking.count():,} lignes")

gold_station_period : 2,334,517 lignes
gold_monthly_overview : 100 lignes
gold_station_ranking : 1,216 lignes


### 3.5. Publication des tables Gold dans PostgreSQL

Les trois tables agrégées sont également chargées dans PostgreSQL afin d'être exploitées par les outils de restitution, notamment le tableau de bord Streamlit.


In [22]:
import commun.spark_utils as su

# Paramètres de connexion à la base PostgreSQL du projet.
su.PG_URL = "jdbc:postgresql://postgres:5432/divvy"
su.PG_PROPS = {
    "user": "divvy",
    "password": "divvy",
    "driver": "org.postgresql.Driver"
}

# Rechargement des tables Delta Gold avant leur publication.
gold_station_period = spark.read.format("delta").load(
    f"{LAKE}/gold/station_period"
)
gold_monthly_overview = spark.read.format("delta").load(
    f"{LAKE}/gold/monthly_overview"
)
gold_station_ranking = spark.read.format("delta").load(
    f"{LAKE}/gold/station_ranking"
)

# Écriture des trois tables dans PostgreSQL.
su.vers_postgres(gold_station_period, "gold_station_period")
su.vers_postgres(gold_monthly_overview, "gold_monthly_overview")
su.vers_postgres(gold_station_ranking, "gold_station_ranking")


Table PostgreSQL « gold_station_period » écrite.
Table PostgreSQL « gold_monthly_overview » écrite.
Table PostgreSQL « gold_station_ranking » écrite.


# 4. Couche sémantique — Ontologie et requêtes SPARQL

## Objectif

La couche sémantique décrit les concepts du domaine Divvy Bike Sharing et leurs relations à travers une ontologie au format Turtle (`.ttl`).

Elle permet de représenter les stations, les périodes, les types de véhicules et les catégories d'utilisateurs, puis d'interroger ces informations à l'aide de requêtes SPARQL.

### 4.1. Chargement des tables Gold pour la couche sémantique

Les données Gold sont rechargées depuis le Data Lake. Elles servent à construire les profils temporels, les répartitions par véhicule et les observations par station.


### Dépendance RDF

La bibliothèque `rdflib` est utilisée pour créer et interroger le graphe sémantique. Cette cellule garantit sa disponibilité dans l'environnement Jupyter.


In [23]:
%pip install rdflib


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 615.4/615.4 kB 6.5 MB/s  0:00:00
Note: you may need to restart the kernel to use updated packages.


In [24]:
from pyspark.sql import functions as F
from rdflib import Graph, Namespace, RDF, Literal, XSD, URIRef
from urllib.parse import quote

gold_station_period = spark.read.format("delta").load(
    f"{LAKE}/gold/station_period"
)
gold_monthly_overview = spark.read.format("delta").load(
    f"{LAKE}/gold/monthly_overview"
)
gold_station_ranking = spark.read.format("delta").load(
    f"{LAKE}/gold/station_ranking"
)


### 4.2. Construction des profils temporels et des véhicules

Les profils sont agrégés à partir des tables Gold afin de représenter les périodes les plus fréquentées et la répartition des trajets selon le véhicule et la catégorie d'utilisateur.


In [25]:
# Profil temporel : volume de trajets par jour de semaine et heure.
rdf_time_profile = (
    gold_station_period
    .groupBy("day_of_week", "hour")
    .agg(
        F.sum("total_trips").alias("total_trips"),
        F.sum("casual_trips").alias("casual_trips"),
        F.sum("member_trips").alias("member_trips")
    )
)

# Répartition par type de véhicule et catégorie d'utilisateur.
rdf_vehicle_profile = (
    gold_monthly_overview
    .groupBy("rideable_type", "member_casual")
    .agg(F.sum("total_trips").alias("total_trips"))
)


### 4.3. Alimentation du graphe RDF

L'ontologie décrit les concepts Divvy. Le graphe est alimenté avec les stations, les observations temporelles et les répartitions par type de véhicule et catégorie d'utilisateur.

L'identifiant de station et son nom sont combinés pour construire les URI : cela permet de distinguer les stations qui partagent un même identifiant dans les données.


In [26]:
# Chargement de l'ontologie et définition des espaces de noms.
g_rdf = Graph()
g_rdf.parse("semantic/ontologie-divvy.ttl", format="turtle")

DIVVY = Namespace("http://example.org/divvy#")
EX = Namespace("http://example.org/divvy/")

g_rdf.bind("divvy", DIVVY)

vehicules = {
    "classic_bike": DIVVY.VeloClassique,
    "electric_bike": DIVVY.VeloElectrique,
    "electric_scooter": DIVVY.ScooterElectrique
}

categories = {
    "casual": DIVVY.Occasionnel,
    "member": DIVVY.Abonne
}

# Stations et statistiques associées.
for row in gold_station_ranking.collect():
    cle_station = f"{row.start_station_id}__{row.start_station_name}"
    station_uri = URIRef(EX + "station/" + quote(cle_station, safe=""))
    observation_uri = URIRef(EX + "observation/station/" + quote(cle_station, safe=""))

    g_rdf.add((station_uri, RDF.type, DIVVY.Station))
    g_rdf.add((station_uri, DIVVY.identifiantStation,
               Literal(row.start_station_id, datatype=XSD.string)))
    g_rdf.add((station_uri, DIVVY.nomStation,
               Literal(row.start_station_name, datatype=XSD.string)))

    g_rdf.add((observation_uri, RDF.type, DIVVY.ObservationTrajets))
    g_rdf.add((observation_uri, DIVVY.concerneStation, station_uri))
    g_rdf.add((observation_uri, DIVVY.nombreTotalTrajets,
               Literal(int(row.total_trips), datatype=XSD.integer)))
    g_rdf.add((observation_uri, DIVVY.nombreTrajetsOccasionnels,
               Literal(int(row.casual_trips), datatype=XSD.integer)))
    g_rdf.add((observation_uri, DIVVY.nombreTrajetsAbonnes,
               Literal(int(row.member_trips), datatype=XSD.integer)))
    g_rdf.add((observation_uri, DIVVY.partOccasionnelle,
               Literal(f"{float(row.casual_share_pct):.2f}", datatype=XSD.decimal)))

# Statistiques temporelles.
for row in rdf_time_profile.collect():
    periode_uri = URIRef(EX + f"periode/jour-{row.day_of_week}/heure-{row.hour}")
    observation_uri = URIRef(EX + f"observation/temps/jour-{row.day_of_week}/heure-{row.hour}")

    g_rdf.add((periode_uri, RDF.type, DIVVY.Periode))
    g_rdf.add((periode_uri, DIVVY.jourSemaine,
               Literal(int(row.day_of_week), datatype=XSD.integer)))
    g_rdf.add((periode_uri, DIVVY.heure,
               Literal(int(row.hour), datatype=XSD.integer)))

    g_rdf.add((observation_uri, RDF.type, DIVVY.ObservationTrajets))
    g_rdf.add((observation_uri, DIVVY.concernePeriode, periode_uri))
    g_rdf.add((observation_uri, DIVVY.nombreTotalTrajets,
               Literal(int(row.total_trips), datatype=XSD.integer)))
    g_rdf.add((observation_uri, DIVVY.nombreTrajetsOccasionnels,
               Literal(int(row.casual_trips), datatype=XSD.integer)))
    g_rdf.add((observation_uri, DIVVY.nombreTrajetsAbonnes,
               Literal(int(row.member_trips), datatype=XSD.integer)))

# Répartition des trajets par véhicule et catégorie.
for row in rdf_vehicle_profile.collect():
    vehicule_uri = vehicules[row.rideable_type]
    categorie_uri = categories[row.member_casual]
    observation_uri = URIRef(
        EX + f"observation/vehicule/{row.rideable_type}/{row.member_casual}"
    )
    repartition_uri = URIRef(
        EX + f"repartition/{row.rideable_type}/{row.member_casual}"
    )

    g_rdf.add((observation_uri, RDF.type, DIVVY.ObservationTrajets))
    g_rdf.add((observation_uri, DIVVY.concerneVehicule, vehicule_uri))
    g_rdf.add((observation_uri, DIVVY.possedeRepartition, repartition_uri))
    g_rdf.add((repartition_uri, RDF.type, DIVVY.RepartitionCategorie))
    g_rdf.add((repartition_uri, DIVVY.concerneCategorie, categorie_uri))
    g_rdf.add((repartition_uri, DIVVY.nombreTrajetsCategorie,
               Literal(int(row.total_trips), datatype=XSD.integer)))


### 4.4. Interrogation du graphe avec SPARQL

Les requêtes suivantes extraient :
- les stations présentant les volumes les plus élevés de trajets occasionnels ;
- les périodes jour/heure les plus fréquentées par les utilisateurs occasionnels ;
- la répartition des trajets par véhicule et catégorie d'utilisateur.


In [27]:
requete_stations = """
PREFIX divvy: <http://example.org/divvy#>

SELECT ?nomStation ?totalTrajets ?trajetsOccasionnels ?partOccasionnelle
WHERE {
    ?observation a divvy:ObservationTrajets ;
                 divvy:concerneStation ?station ;
                 divvy:nombreTotalTrajets ?totalTrajets ;
                 divvy:nombreTrajetsOccasionnels ?trajetsOccasionnels ;
                 divvy:partOccasionnelle ?partOccasionnelle .

    ?station divvy:nomStation ?nomStation .
}
ORDER BY DESC(?trajetsOccasionnels)
LIMIT 10
"""

resultat_stations = g_rdf.query(requete_stations)

print(f"{'Station':<40} {'Total':>10} {'Occasionnels':>15} {'Part (%)':>10}")
print("-" * 80)

for ligne in resultat_stations:
    print(
        f"{str(ligne.nomStation):<40} "
        f"{int(ligne.totalTrajets):>10,} "
        f"{int(ligne.trajetsOccasionnels):>15,} "
        f"{float(ligne.partOccasionnelle):>10.2f}"
    )

Station                                       Total    Occasionnels   Part (%)
--------------------------------------------------------------------------------
Streeter Dr & Grand Ave                      77,320          59,515      76.97
DuSable Lake Shore Dr & Monroe St            52,493          40,829      77.78
Michigan Ave & Oak St                        46,293          28,851      62.32
Navy Pier                                    35,135          27,414      78.02
Millennium Park                              39,542          26,765      67.69
DuSable Lake Shore Dr & North Blvd           45,360          26,111      57.56
Shedd Aquarium                               30,976          25,157      81.21
DuSable Lake Shore Dr & Monroe St            31,105          24,603      79.10
Dusable Harbor                               30,102          22,128      73.51
Theater on the Lake                          34,373          19,271      56.06


In [28]:
requete_periodes = """
PREFIX divvy: <http://example.org/divvy#>

SELECT ?jour ?heure ?trajetsOccasionnels ?totalTrajets
WHERE {
    ?observation a divvy:ObservationTrajets ;
                 divvy:concernePeriode ?periode ;
                 divvy:nombreTrajetsOccasionnels ?trajetsOccasionnels ;
                 divvy:nombreTotalTrajets ?totalTrajets .

    ?periode divvy:jourSemaine ?jour ;
             divvy:heure ?heure .
}
ORDER BY DESC(?trajetsOccasionnels)
LIMIT 10
"""

resultat_periodes = g_rdf.query(requete_periodes)

jours = {
    1: "Dimanche",
    2: "Lundi",
    3: "Mardi",
    4: "Mercredi",
    5: "Jeudi",
    6: "Vendredi",
    7: "Samedi"
}

print(f"{'Jour':<15} {'Heure':>8} {'Occasionnels':>18} {'Total':>15}")
print("-" * 62)

for ligne in resultat_periodes:
    print(
        f"{jours[int(ligne.jour)]:<15} "
        f"{int(ligne.heure):02d}h{'':<5} "
        f"{int(ligne.trajetsOccasionnels):>18,} "
        f"{int(ligne.totalTrajets):>15,}"
    )

Jour               Heure       Occasionnels           Total
--------------------------------------------------------------
Samedi          15h                  58,566         113,265
Samedi          14h                  58,517         114,127
Samedi          13h                  57,964         114,271
Samedi          16h                  56,371         110,958
Samedi          12h                  56,008         112,400
Dimanche        15h                  51,013         102,224
Vendredi        17h                  50,490         136,587
Samedi          11h                  50,358         104,634
Samedi          17h                  49,849         101,000
Dimanche        14h                  49,606         100,128


In [29]:
requete_vehicules = """
PREFIX divvy: <http://example.org/divvy#>

SELECT ?vehicule ?categorie ?nombreTrajets
WHERE {
    ?observation a divvy:ObservationTrajets ;
                 divvy:concerneVehicule ?vehicule ;
                 divvy:possedeRepartition ?repartition .

    ?repartition divvy:concerneCategorie ?categorie ;
                 divvy:nombreTrajetsCategorie ?nombreTrajets .
}
ORDER BY ?vehicule ?categorie
"""

resultat_vehicules = g_rdf.query(requete_vehicules)

noms_vehicules = {
    DIVVY.VeloClassique: "Vélo classique",
    DIVVY.VeloElectrique: "Vélo électrique",
    DIVVY.ScooterElectrique: "Scooter électrique"
}

noms_categories = {
    DIVVY.Occasionnel: "Occasionnel",
    DIVVY.Abonne: "Abonné"
}

print(f"{'Véhicule':<22} {'Catégorie':<18} {'Nombre de trajets':>20}")
print("-" * 62)

for ligne in resultat_vehicules:
    print(
        f"{noms_vehicules[ligne.vehicule]:<22} "
        f"{noms_categories[ligne.categorie]:<18} "
        f"{int(ligne.nombreTrajets):>20,}"
    )

Véhicule               Catégorie             Nombre de trajets
--------------------------------------------------------------
Scooter électrique     Abonné                           59,122
Scooter électrique     Occasionnel                      85,215
Vélo classique         Abonné                        3,035,963
Vélo classique         Occasionnel                   1,647,524
Vélo électrique        Abonné                        4,166,801
Vélo électrique        Occasionnel                   2,418,046


### 4.5. Sauvegarde du graphe RDF

Le graphe final est exporté au format Turtle dans le dossier `semantic`. Le fichier pourra être réutilisé sans reconstruire les triplets.


In [30]:
from pathlib import Path

Path("semantic").mkdir(exist_ok=True)

g_rdf.serialize(
    destination="semantic/graphe-divvy.ttl",
    format="turtle"
)


<Graph identifier=N545c60bccdf04abbb4e99d5602772feb (<class 'rdflib.graph.Graph'>)>

In [31]:
verification = Graph()
verification.parse("semantic/graphe-divvy.ttl", format="turtle")

print(f"Nombre de triples RDF : {len(verification):,}")

Nombre de triples RDF : 12,403
